# 10 · P2 Data Understanding — 7개년(2019~2025) 데이터 이해

> CRISP-DM **P2(Data Understanding)**. 마스터플랜([`docs/design/analysis-master-plan.md`](../docs/design/analysis-master-plan.md)) §1·§2가 "발견"이라고 적은 7개년 정합성 난제(인코딩 3종·가중치명 4종·변수명 직결 불가·2022 표본 이상)를 **코드로 직접 재현**해 입증한다.
>
> **thin 원칙**: 로직 정본은 `src/`다. 본 노트북은 `src/harmonize.py`·`src/extract_all_sav_meta.py`의 단계함수·상수를 import해 *과정과 중간 출력*을 보여줄 뿐, 로직을 재구현하지 않는다.
>
> ⚠️ **검증 게이트**([`docs/design/data-spec.md`](../docs/design/data-spec.md) §6): 아래 수치는 데이터 이해용이며 KPF 원자료 재검증 전 보고서·웹데모 직접 인용 금지(방향·구조 해석 위주).

**종합 문서**: [data-spec.md](../docs/design/data-spec.md) · [variable-crosswalk.md](../docs/design/variable-crosswalk.md) · [variable-crosswalk-trust-battery.md](../docs/design/variable-crosswalk-trust-battery.md) · [sav-meta-catalog.md](../docs/design/sav-meta-catalog.md)

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import pyreadstat

# 저장소 루트 탐색(노트북 cwd가 notebooks/든 루트든 견고)
ROOT = Path.cwd()
while not (ROOT / "src" / "harmonize.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import harmonize as hz                 # P3 SSOT — 상수(SRC, CRED_BATTERY)·로더 재사용
print("ROOT =", ROOT)
print("대상 연도 =", hz.YEARS, "| T(균등화 분모) =", hz.T)

ROOT = C:\Users\kik32\workspace\Dacon\Media-Statistics-Analysis-and-Utilization-Competition
대상 연도 = [2019, 2020, 2021, 2022, 2023, 2024, 2025] | T(균등화 분모) = 7


## 1. 7개년 메타 실측 — N·변수수·인코딩·가중치

마스터플랜 §1 표를 *말이 아니라 코드로* 재현한다. 속도를 위해 `metadataonly=True`로
헤더만 읽되(전체 로드 불필요), `src`와 동일한 **인코딩 fallback(euc-kr→utf-8)** 정책을 그대로 적용한다.

In [2]:
def read_meta(path: Path):
    # 인코딩 순차 시도(euc-kr→utf-8)로 메타만 로드. src/harmonize.read_sav_any와 동일 정책.
    last = None
    for enc in hz.ENCODINGS:                 # ['euc-kr','utf-8'] — src 상수 재사용
        try:
            _, meta = pyreadstat.read_sav(str(path), metadataonly=True, encoding=enc)
            return meta, enc
        except Exception as e:
            last = e
    raise RuntimeError(f"{path.name}: 모든 인코딩 실패 → {last}")

WEIGHT_CANDS = {"wt1", "wt2", "WT", "HMWT", "wt", "hmwt"}
rows, META = [], {}
for y in hz.YEARS:
    meta, enc = read_meta(hz.SAV_BY_YEAR[y])
    META[y] = (meta, enc)
    names = list(meta.column_names)
    wts = [n for n in names if n in WEIGHT_CANDS]
    rows.append({"연도": y, "표본 N": meta.number_rows, "변수 수": meta.number_columns,
                 "인코딩": enc, "가중치 변수": ", ".join(wts)})
meta_tbl = pd.DataFrame(rows).set_index("연도")
meta_tbl

,표본 N,변수 수,인코딩,가중치 변수
연도,,,,
2019,5040,286,euc-kr,"wt1, wt2"
2020,5010,281,euc-kr,WT
2021,5010,307,utf-8,WT
2022,58936,244,utf-8,WT
2023,5000,245,euc-kr,HMWT
2024,6000,237,euc-kr,WT
2025,6000,223,euc-kr,WT


**관찰**:
- **인코딩 혼재** — euc-kr(2019·20·23·24·25) vs **utf-8(2021·2022)**. 단일 인코딩 로더가 2021에서 깨지므로 fallback이 필수다(아래 §1.1에서 fallback이 실제로 작동함을 입증).
- **가중치 변수명 4종** — `wt1/wt2`(2019)·`WT`(20·21·22·24·25)·`HMWT`(2023). → `src/harmonize.WEIGHT_BY_YEAR`로 표준 `WT` 통일.
- **2022 표본 이상** — N이 타 연도(5천~6천) 대비 압도적. 개인용 파일의 구조를 §3에서 규명.

In [3]:
# §1.1 인코딩 fallback 입증: 2021(utf-8)을 euc-kr로 강제하면 실제로 예외가 나는가?
p2021 = hz.SAV_BY_YEAR[2021]
try:
    pyreadstat.read_sav(str(p2021), metadataonly=True, encoding="euc-kr")
    print("2021 euc-kr: 예외 없음(주의 — 무성 모자이크 가능성 점검 필요)")
except Exception as e:
    print("2021 euc-kr → 예외 발생 → fallback이 utf-8로 정상 전환:", type(e).__name__)
print("실측 인코딩(2021) =", META[2021][1])
# 라벨이 깨지지 않았는지(한글 정상) 확인 — 2025 인구통계 라벨 샘플
m25 = META[2025][0]
sample = {k: m25.column_names_to_labels.get(k) for k in list(m25.column_names)[:6]}
print("2025 라벨 샘플(한글 정상 디코드 확인):")
for k, v in sample.items():
    print(f"  {k}: {v}")

2021 euc-kr → 예외 발생 → fallback이 utf-8로 정상 전환: ReadstatError
실측 인코딩(2021) = utf-8
2025 라벨 샘플(한글 정상 디코드 확인):
  ID: 응답자 개별 ID
  SQ1: SQ1. 거주지역_시도
  DQ2: DQ2. 성별
  DQ3: DQ3. 연령
  Q1: 문1. 지난 일주일 간 종이신문 열독 여부
  Q2A_1: 문2A. 종이신문 열독 일수_평일


## 2. 변수명 직결 불가 — 문항번호 체계가 2~3년마다 리셋

7개년을 변수명으로 그냥 이어붙일 수 있는가? **인접연도 Jaccard 유사도**와 **7개년 공통 교집합 크기**로 확인한다.

In [4]:
namesets = {y: set(META[y][0].column_names) for y in hz.YEARS}
# 인접연도 Jaccard
jac = []
for a, b in zip(hz.YEARS[:-1], hz.YEARS[1:]):
    inter = len(namesets[a] & namesets[b]); uni = len(namesets[a] | namesets[b])
    jac.append({"구간": f"{a}-{b}", "공통 변수": inter, "Jaccard": round(inter/uni, 3)})
jac_tbl = pd.DataFrame(jac).set_index("구간")
common_all = set.intersection(*namesets.values())
print("7개년 공통 변수명 교집합:", len(common_all), "개 →", sorted(common_all))
jac_tbl

7개년 공통 변수명 교집합: 2 개 → ['Q1', 'SQ1']


,공통 변수,Jaccard
구간,,
2019-2020,44,0.084
2020-2021,248,0.729
2021-2022,52,0.104
2022-2023,53,0.122
2023-2024,200,0.709
2024-2025,142,0.447


**관찰**: 7개년 공통 변수명은 **단 2개**, 인접연도 Jaccard도 0.10~0.73으로 들쭉날쭉하다. → 변수명을 직결할 수 없고, **문항 의미 기반 crosswalk(매핑 테이블)** 가 필수임이 코드로 확인된다. 그 결과물이 [`variable-crosswalk.md`](../docs/design/variable-crosswalk.md)·[`-trust-battery.md`](../docs/design/variable-crosswalk-trust-battery.md)이며, `src/harmonize.SRC`·`CRED_BATTERY` 상수로 코드화돼 있다(§4).

## 3. 2022 표본 이상 규명 — 가구용 vs 개인용

2022 디렉터리에는 `가구용`·`개인용` 두 `.sav`가 있다. 분석단위(개인)에 맞는 파일이 무엇이고 N=58,936이 어디서 오는지 확인한다.

In [5]:
AUD = hz.AUD / "2022"
for tag, fn in [("가구용", "2022_언론수용자조사_가구용_데이터.sav"),
                ("개인용", "2022_언론수용자조사_개인용_데이터.sav")]:
    m, enc = read_meta(AUD / fn)
    print(f"{tag}: N={m.number_rows:,}  변수={m.number_columns}  enc={enc}")
print("\nsrc/harmonize가 2022에 채택한 파일:", hz.SAV_BY_YEAR[2022].name)
print("→ 분석단위=개인 → 개인용(N=58,936) 사용. 가구용은 가구 레벨이라 개인 반복횡단면에 부적합.")

가구용: N=30,138  변수=14  enc=utf-8
개인용: N=58,936  변수=244  enc=utf-8

src/harmonize가 2022에 채택한 파일: 2022_언론수용자조사_개인용_데이터.sav
→ 분석단위=개인 → 개인용(N=58,936) 사용. 가구용은 가구 레벨이라 개인 반복횡단면에 부적합.


**결론**: 2022 개인용 N=58,936은 오류가 아니라 **대규모 개인 표본**이다(타 연도의 ~10배). 이는 추세·평균에서 2022가 표본을 지배하게 하므로, P3에서 **연도기여 균등 가중치(`wt_year_eq`)** 로 보정한다(노트북 11 §3).

## 4. 신뢰성 배터리 crosswalk — 핵심 3지표는 7개년 전부 존재

측정 비동등 검정(MGCFA)·정렬법의 잠재요인 입력이 되는 **credibility 다지표 배터리**의 7개년 가용성을, `src/harmonize.CRED_BATTERY` 상수에서 직접 읽어 표로 만든다(문항 prefix는 연도마다 리셋되지만 의미는 동일).

In [6]:
def avail_table(mapping: dict) -> pd.DataFrame:
    rows = []
    for target, ymap in mapping.items():
        row = {"target": target}
        for y in hz.YEARS:
            row[y] = ymap.get(y) or "—"
        rows.append(row)
    return pd.DataFrame(rows).set_index("target")

cred_tbl = avail_table(hz.CRED_BATTERY)
print("핵심 3지표(주 모형):", hz.CRED_FACTOR_CORE3)
print("4지표(민감도, +신뢰):", hz.CRED_FACTOR_PLUS4)
cred_tbl

핵심 3지표(주 모형): ['cred_fair', 'cred_professional', 'cred_accurate']
4지표(민감도, +신뢰): ['cred_fair', 'cred_professional', 'cred_accurate', 'cred_trustworthy']


,2019,2020,2021,2022,2023,2024,2025
target,,,,,,,
cred_fair,Q77_1,Q78_1,Q78_1,Q69_1,Q77_1,Q77_1,Q85_1
cred_professional,Q77_2,Q78_2,Q78_2,Q69_2,Q77_2,Q77_2,Q85_2
cred_accurate,Q77_3,Q78_3,Q78_3,Q69_3,Q77_3,Q77_3,Q85_3
cred_trustworthy,Q77_4,Q78_4,Q78_4,Q69_4,—,—,—
press_free,Q77_5,Q78_5,Q78_5,Q69_5,Q77_4,Q77_4,Q85_4
media_influence,—,—,Q78_6,Q69_6,Q77_5,Q77_5,Q85_5


In [7]:
# 연도별 '신뢰성 요인 지표 가용 개수' 요약
core_avail = {y: sum(hz.CRED_BATTERY[i][y] is not None for i in hz.CRED_FACTOR_CORE3) for y in hz.YEARS}
plus_avail = {y: sum(hz.CRED_BATTERY[i][y] is not None for i in hz.CRED_FACTOR_PLUS4) for y in hz.YEARS}
pd.DataFrame({"핵심3지표 가용수": core_avail, "4지표(+신뢰) 가용수": plus_avail}).T

,2019,2020,2021,2022,2023,2024,2025
핵심3지표 가용수,3,3,3,3,3,3,3
4지표(+신뢰) 가용수,4,4,4,4,3,3,3


**관찰**: 핵심 3지표{공정·전문·정확}은 **7개년 전부(=3)** 존재 → 잠재 신뢰성 요인을 2019~2025로 검정 가능. 신뢰 직접지표(`cred_trustworthy`)는 2019~2022만 → 4지표는 민감도 모형으로만 사용. 단일 요약문항(`trust_news_overall`)은 2019 부재(2020~)라 추세가 짧다(§5). 이 구조가 "단일문항은 2020~, 다지표 잠재요인은 2019~"라는 crosswalk-trust-battery의 핵심 정밀화다.

## 5. 핵심지표 가용성 행렬 — 결측은 '무응답'이 아니라 '구조적 부재'

`src/harmonize.SRC`(인구·신뢰·매체)와 `CRED_BATTERY`를 합쳐, 연도×target의 **원천변수 존재 여부**(설계상 가용성)를 행렬로 본다. 빈칸은 무작위 무응답이 아니라 **문항 도입/폐지에 의한 구조적 부재**다.

In [8]:
struct = {**{k: v for k, v in hz.SRC.items()}, **hz.CRED_BATTERY}
mat = avail_table(struct)
# 존재=●, 부재=· 로 표기 (pandas 버전 무관: apply+map)
disp = mat.apply(lambda s: s.map(lambda x: "·" if x == "—" else "●"))
disp

,2019,2020,2021,2022,2023,2024,2025
target,,,,,,,
trust_news_overall,·,●,●,●,●,●,●
trust_news_used,·,·,●,●,●,●,●
trust_society,·,●,●,●,●,●,●
media_main_route,●,●,●,●,●,●,●
sex,●,●,●,●,●,●,●
age,●,●,●,●,●,●,●
edu,●,●,●,●,●,●,●
income,●,●,●,●,●,●,●
job,●,●,●,●,●,●,●


**관찰**: `trust_news_overall`(2019 부재)·`trust_news_used`(2019·2020 부재)·`cred_trustworthy`(2023~ 부재)·`media_influence`(2019·2020 부재)의 빈칸은 전부 **문항 도입/폐지** 때문이다. 따라서 결측 처리는 '대치'가 아니라 '구조적 부재로 명시'가 원칙(P3).

## 6. 검증 셀 — docs 수치와 일치하는가 (입증)

In [9]:
EXPECT_N = {2019: 5040, 2020: 5010, 2021: 5010, 2022: 58936, 2023: 5000, 2024: 6000, 2025: 6000}
EXPECT_ENC = {2019: "euc-kr", 2020: "euc-kr", 2021: "utf-8", 2022: "utf-8",
              2023: "euc-kr", 2024: "euc-kr", 2025: "euc-kr"}
for y in hz.YEARS:
    n, enc = META[y][0].number_rows, META[y][1]
    assert n == EXPECT_N[y], f"{y} N 불일치: {n} != {EXPECT_N[y]}"
    assert enc == EXPECT_ENC[y], f"{y} 인코딩 불일치: {enc} != {EXPECT_ENC[y]}"
assert len(common_all) == 2, f"7개년 공통 변수명 != 2 (={len(common_all)})"
assert all(core_avail[y] == 3 for y in hz.YEARS), "핵심3지표가 7개년 전부 가용이 아님"
print("PASS — 연도별 N·인코딩·공통변수(2개)·핵심3지표 7개년 가용 모두 docs와 일치.")

PASS — 연도별 N·인코딩·공통변수(2개)·핵심3지표 7개년 가용 모두 docs와 일치.


## 7. 결론 & 다음 단계

- 마스터플랜이 '발견'이라 적은 4난제(인코딩 3종·가중치명 4종·변수명 직결 불가·2022 대표본)를 **코드로 재현·입증**했다.
- 신뢰성 잠재요인의 7개년 가용성(핵심3지표=7개년 전부)을 확인 → MGCFA/정렬법의 입력 정당성 토대.
- **코드리뷰(`extract_all_sav_meta.py`)**: 메타 추출 로직 정확. 단, 원 스크립트는 전체 로드 후 메타만 사용 → 본 노트북처럼 `metadataonly=True`가 더 효율적(기능 동일, 성능 개선 여지 — 기록만, 강제수정 불필요).
- **다음**: [`11-data-prep-harmonize.ipynb`](11-data-prep-harmonize.ipynb) — 위 crosswalk를 적용해 `src/harmonize.build_year`로 90,996행 패널을 단계별로 빌드·검증.